# MM-Fit Feature Extraction Pipeline

This notebook mirrors the UTD-MHAD feature extraction pipeline but is adapted for the **MM-Fit** dataset.

**Modalities used:** Skeleton (pose_3d as primary, pose_2d as fallback) + all sensor streams (accelerometer, gyroscope, magnetometer, heart-rate).

**Key adaptations from UTD-MHAD:**
- Skeleton joints remapped from Kinect-20 to Human3.6M-17 (pose_3d) or COCO-18 (pose_2d)
- `HAND_LEFT` / `HAND_RIGHT` and `FOOT_LEFT` / `FOOT_RIGHT` are approximated from wrist/ankle since H3.6M has no dedicated hand/foot joints
- Inertial data comes from multiple sensor streams (smartwatch L/R acc+gyr, smartphone L/R acc+gyr+mag, earbud acc+gyr) which are each extracted and concatenated
- Labels are segment-based: each (start_frame, end_frame, rep_count, activity_class) entry becomes one sample
- Train/test split follows leave-one-participant-out (participants 0-9)

## Section 1: Imports & Configuration

In [1]:
import os
import sys
import glob
import warnings
import time
import numpy as np
import pandas as pd
from scipy import stats, signal
from scipy.ndimage import gaussian_filter1d
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import cross_val_score
import logging
import joblib

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info('Imports complete.')

2026-05-31 17:07:04,725 - INFO - Imports complete.


In [2]:
class Config:
    """Central configuration for the MM-Fit pipeline."""

    # ── Dataset ──────────────────────────────────────────────────────────────
    DATASET_ROOT = './mm-fit'          # folder containing w00 … w20
    WORKOUT_IDS  = [f'w{i:02d}' for i in range(21)]   # w00 to w20

    # Workout → participant mapping (from MM-Fit documentation)
    WORKOUT_TO_PARTICIPANT = {
        'w00': 2, 'w01': 0, 'w02': 1, 'w03': 0, 'w04': 1,
        'w05': 2, 'w06': 0, 'w07': 1, 'w08': 0, 'w09': 1,
        'w10': 0, 'w11': 1, 'w12': 3, 'w13': 4, 'w14': 0,
        'w15': 1, 'w16': 5, 'w17': 6, 'w18': 7, 'w19': 8, 'w20': 9,
    }

    # ── Output ───────────────────────────────────────────────────────────────
    FEATURES_DIR = 'features_noise_1.0_50'

    # ── Skeleton ─────────────────────────────────────────────────────────────
    # Primary skeleton: pose_3d  (Human3.6M, 17 joints, shape 3×N×33 — only 17 used)
    # Fallback skeleton: pose_2d (COCO, 18 joints, shape 2×N×19 — only 18 used)
    USE_POSE_3D  = True   # preferred
    USE_POSE_2D  = True   # used when pose_3d absent

    # ── Sensor streams to use ────────────────────────────────────────────────
    SENSOR_STREAMS = [
        'sw_l_acc', 'sw_l_gyr',
        'sw_r_acc', 'sw_r_gyr',
        'sp_l_acc', 'sp_l_gyr',
        'sp_r_acc', 'sp_r_gyr',
        'eb_l_acc', 'eb_l_gyr',
    ]
    # Heart-rate streams (single channel; extracted separately)
    HR_STREAMS = ['sw_l_hr', 'sw_r_hr']
    # Magnetometer streams (3-channel like acc/gyr)
    MAG_STREAMS = ['sp_l_mag', 'sp_r_mag']

    # ── Sensor feature-extraction parameters ─────────────────────────────────
    SENSOR_WINDOW_SIZE    = 50
    SENSOR_WINDOW_OVERLAP = 0.5

    # ── Train / test split ────────────────────────────────────────────────────
    # Leave-one-participant-out or fixed split:
    # participants 0-7 train, 8-9 test  (adjust as needed)
    TRAIN_PARTICIPANTS = [0, 1, 2, 3, 4, 5, 6, 7]
    TEST_PARTICIPANTS  = [8, 9]

    # ── Random Forest ─────────────────────────────────────────────────────────
    RF_N_ESTIMATORS      = 300
    RF_MAX_DEPTH         = None
    RF_MIN_SAMPLES_SPLIT = 2
    RF_MIN_SAMPLES_LEAF  = 1
    RF_RANDOM_STATE      = 42
    RF_N_JOBS            = 1

config = Config()
logger.info('Config ready.')

2026-05-31 17:07:04,735 - INFO - Config ready.


## Section 2: Data Loading

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Lightweight utils (mimics mm-fit's utils.py so the notebook is self-contained)
# ─────────────────────────────────────────────────────────────────────────────

def load_modality(path):
    """Load a modality .npy file; return None if file missing or unreadable."""
    if not os.path.isfile(path):
        return None
    try:
        return np.load(path, allow_pickle=False)
    except Exception as e:
        logger.debug(f'Cannot load {path}: {e}')
        return None


def load_labels(path):
    """
    Load labels CSV.
    Columns: start_frame, end_frame, rep_count, activity_class
    Returns list of [start_frame, end_frame, rep_count, activity_class] rows.
    """
    if not os.path.isfile(path):
        return []
    try:
        df = pd.read_csv(path, header=None)
        return df.values.tolist()
    except Exception as e:
        logger.debug(f'Cannot load labels {path}: {e}')
        return []


logger.info('Utils defined.')

2026-05-31 17:07:04,745 - INFO - Utils defined.


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# MM-Fit Dataset Loader
# ─────────────────────────────────────────────────────────────────────────────

class MMFitLoader:
    """
    Loads all modalities and labels for each workout folder (w00–w20).

    Each workout is segmented into activity samples using the labels CSV.
    Each label entry (start_frame, end_frame, rep_count, activity_class)
    defines one sample.  Modality data is sliced to the frame range using
    the Frame column (column 0) of each array.

    Returns a flat list of sample dicts.
    """

    def __init__(self, config):
        self.config = config
        self.root   = config.DATASET_ROOT

    # ── helpers ───────────────────────────────────────────────────────────────

    @staticmethod
    def _slice_by_frame(array, start_frame, end_frame):
        """
        Slice a (N, …) modality array to rows whose Frame column is in
        [start_frame, end_frame].  Column 0 is always the Frame index.
        Returns the data columns only (drops the Frame column).
        Returns None if the resulting slice is empty.
        """
        if array is None or array.ndim < 2:
            return None
        frames = array[:, 0].astype(int)
        mask   = (frames >= start_frame) & (frames <= end_frame)
        sliced = array[mask]
        if sliced.shape[0] == 0:
            return None
        return sliced[:, 1:]   # drop Frame column; keep Timestamp + data

    @staticmethod
    def _slice_pose_by_frame(pose, start_frame, end_frame):
        """
        Slice pose arrays.  MM-Fit pose shape: (dims, N, joints+1) where
        axis-1 is samples and the first value along axis-2 is the Frame index.

        pose_2d: (2, N, 19)  → row 0 per dim is frame,  joints 1-18
        pose_3d: (3, N, 33)  → row 0 per dim is frame,  joints 1-32

        We use dim-0 (x) to extract the frame index (same across dims).
        Returns array of shape (n_sliced, n_joints, n_dims) or None.
        """
        if pose is None:
            return None
        if pose.ndim != 3:
            return None
        n_dims, n_samples, n_cols = pose.shape
        # Frame indices stored in first column of each dimension (identical across dims)
        frames = pose[0, :, 0].astype(int)
        mask   = (frames >= start_frame) & (frames <= end_frame)
        sliced = pose[:, mask, :]   # (n_dims, n_sliced, n_cols)
        if sliced.shape[1] == 0:
            return None
        # Drop frame-index column → (n_dims, n_sliced, n_joints)
        sliced = sliced[:, :, 1:]
        # Reshape to (n_sliced, n_joints, n_dims)
        sliced = np.transpose(sliced, (1, 2, 0))  # (n_sliced, n_joints, n_dims)
        return sliced

    # ── loading ───────────────────────────────────────────────────────────────

    def load_workout(self, w_id):
        """Load all modalities and labels for one workout folder."""
        data_dir = os.path.join(self.root, w_id)

        def path(suffix):
            return os.path.join(data_dir, f'{w_id}_{suffix}')

        raw = {}
        for stream in (self.config.SENSOR_STREAMS
                       + self.config.HR_STREAMS
                       + self.config.MAG_STREAMS):
            raw[stream] = load_modality(path(f'{stream}.npy'))

        raw['pose_2d'] = load_modality(path('pose_2d.npy'))
        raw['pose_3d'] = load_modality(path('pose_3d.npy'))

        labels = load_labels(path('labels.csv'))
        return raw, labels

    def load_all(self):
        """
        Iterate over all workout folders and produce a flat list of samples.

        Each sample dict:
            workout_id, participant_id, activity_class, rep_count,
            start_frame, end_frame,
            pose           → (frames, joints, dims) or None
            pose_type      → '3d' | '2d' | None
            sensor_streams → {stream_name: (N, data_cols) or None}
        """
        samples = []
        missing_workouts = []

        for w_id in self.config.WORKOUT_IDS:
            workout_dir = os.path.join(self.root, w_id)
            if not os.path.isdir(workout_dir):
                missing_workouts.append(w_id)
                continue

            participant = self.config.WORKOUT_TO_PARTICIPANT[w_id]
            raw, labels = self.load_workout(w_id)

            if not labels:
                logger.warning(f'{w_id}: no labels found, skipping.')
                continue

            for row in labels:
                start_frame  = int(row[0])
                end_frame    = int(row[1])
                rep_count    = int(row[2])
                activity_cls = row[3]

                # ── pose ─────────────────────────────────────────────────────
                pose      = None
                pose_type = None

                if self.config.USE_POSE_3D and raw.get('pose_3d') is not None:
                    p = self._slice_pose_by_frame(
                        raw['pose_3d'], start_frame, end_frame)
                    if p is not None and p.shape[0] > 0:
                        pose      = p
                        pose_type = '3d'

                if pose is None and self.config.USE_POSE_2D and raw.get('pose_2d') is not None:
                    p = self._slice_pose_by_frame(
                        raw['pose_2d'], start_frame, end_frame)
                    if p is not None and p.shape[0] > 0:
                        pose      = p
                        pose_type = '2d'

                # ── sensor streams ───────────────────────────────────────────
                sensor_streams = {}
                all_sensor_keys = (self.config.SENSOR_STREAMS
                                   + self.config.HR_STREAMS
                                   + self.config.MAG_STREAMS)
                for stream in all_sensor_keys:
                    sensor_streams[stream] = self._slice_by_frame(
                        raw.get(stream), start_frame, end_frame)

                samples.append({
                    'workout_id':    w_id,
                    'participant':   participant,
                    'activity_class': activity_cls,
                    'rep_count':     rep_count,
                    'start_frame':   start_frame,
                    'end_frame':     end_frame,
                    'pose':          pose,
                    'pose_type':     pose_type,
                    'sensor_streams': sensor_streams,
                })

        if missing_workouts:
            logger.warning(f'Missing workout folders: {missing_workouts}')
        logger.info(f'Total samples loaded: {len(samples)}')
        return samples


logger.info('MMFitLoader defined.')

2026-05-31 17:07:04,762 - INFO - MMFitLoader defined.


## Section 3: EDA — Verify Data Loading

In [5]:
loader  = MMFitLoader(config)
samples = loader.load_all()

print(f'\nTotal segments/samples: {len(samples)}')
if samples:
    print(f'Keys in a sample dict:  {list(samples[0].keys())}')

2026-05-31 17:07:19,029 - INFO - Total samples loaded: 616



Total segments/samples: 616
Keys in a sample dict:  ['workout_id', 'participant', 'activity_class', 'rep_count', 'start_frame', 'end_frame', 'pose', 'pose_type', 'sensor_streams']


In [6]:
# ─── Summary statistics ───────────────────────────────────────────────────────
import collections

activity_counts     = collections.Counter(s['activity_class'] for s in samples)
participant_counts  = collections.Counter(s['participant']     for s in samples)
pose_type_counts    = collections.Counter(s['pose_type']       for s in samples)

print('=== Activity class distribution ===')
for cls, cnt in sorted(activity_counts.items()):
    print(f'  Class {cls}: {cnt:4d} samples')

print('\n=== Participant distribution ===')
for pid, cnt in sorted(participant_counts.items()):
    print(f'  Participant {pid}: {cnt} samples')

print('\n=== Pose type availability ===')
for pt, cnt in pose_type_counts.items():
    print(f'  {pt}: {cnt} samples')

=== Activity class distribution ===
  Class bicep_curls:   59 samples
  Class dumbbell_rows:   64 samples
  Class dumbbell_shoulder_press:   60 samples
  Class jumping_jacks:   57 samples
  Class lateral_shoulder_raises:   56 samples
  Class lunges:   62 samples
  Class pushups:   65 samples
  Class situps:   65 samples
  Class squats:   64 samples
  Class tricep_extensions:   64 samples

=== Participant distribution ===
  Participant 0: 184 samples
  Participant 1: 170 samples
  Participant 2: 60 samples
  Participant 3: 22 samples
  Participant 4: 30 samples
  Participant 5: 33 samples
  Participant 6: 34 samples
  Participant 7: 30 samples
  Participant 8: 26 samples
  Participant 9: 27 samples

=== Pose type availability ===
  3d: 616 samples


In [7]:
# ─── Inspect shapes of the first sample with pose data ────────────────────────
sample_with_pose = next((s for s in samples if s['pose'] is not None), None)

if sample_with_pose:
    print(f'Workout:        {sample_with_pose["workout_id"]}')
    print(f'Participant:    {sample_with_pose["participant"]}')
    print(f'Activity class: {sample_with_pose["activity_class"]}')
    print(f'Frames:         [{sample_with_pose["start_frame"]}, {sample_with_pose["end_frame"]}]')
    print(f'Pose type:      {sample_with_pose["pose_type"]}')
    print(f'Pose shape:     {sample_with_pose["pose"].shape}  (frames, joints, dims)')
    print()
    for stream, arr in sample_with_pose['sensor_streams'].items():
        shape = arr.shape if arr is not None else 'None'
        print(f'  {stream:12s}: {shape}')
else:
    print('No sample with pose data found — check dataset path.')

Workout:        w00
Participant:    2
Activity class: squats
Frames:         [4040, 4500]
Pose type:      3d
Pose shape:     (451, 17, 3)  (frames, joints, dims)

  sw_l_acc    : (1549, 4)
  sw_l_gyr    : (1549, 4)
  sw_r_acc    : (1564, 4)
  sw_r_gyr    : (1564, 4)
  sp_l_acc    : None
  sp_l_gyr    : None
  sp_r_acc    : (3260, 4)
  sp_r_gyr    : (3259, 4)
  eb_l_acc    : (1432, 4)
  eb_l_gyr    : (1432, 4)
  sw_l_hr     : (14, 2)
  sw_r_hr     : (14, 2)
  sp_l_mag    : None
  sp_r_mag    : (1507, 4)


In [8]:
# ─── Missing-data audit ───────────────────────────────────────────────────────
all_streams = config.SENSOR_STREAMS + config.HR_STREAMS + config.MAG_STREAMS
stream_missing = {s: 0 for s in all_streams}
stream_missing['pose'] = 0
n = len(samples)

for s in samples:
    if s['pose'] is None:
        stream_missing['pose'] += 1
    for st in all_streams:
        if s['sensor_streams'].get(st) is None:
            stream_missing[st] += 1

print(f'Missing data audit (out of {n} samples):')
for k, v in stream_missing.items():
    pct = 100 * v / n if n else 0
    print(f'  {k:15s}: {v:4d} missing  ({pct:.1f}%)')

Missing data audit (out of 616 samples):
  sw_l_acc       :    0 missing  (0.0%)
  sw_l_gyr       :    0 missing  (0.0%)
  sw_r_acc       :    0 missing  (0.0%)
  sw_r_gyr       :    0 missing  (0.0%)
  sp_l_acc       :  616 missing  (100.0%)
  sp_l_gyr       :  616 missing  (100.0%)
  sp_r_acc       :    0 missing  (0.0%)
  sp_r_gyr       :    0 missing  (0.0%)
  eb_l_acc       :    0 missing  (0.0%)
  eb_l_gyr       :    0 missing  (0.0%)
  sw_l_hr        :  100 missing  (16.2%)
  sw_r_hr        :  207 missing  (33.6%)
  sp_l_mag       :  616 missing  (100.0%)
  sp_r_mag       :    0 missing  (0.0%)
  pose           :    0 missing  (0.0%)


In [9]:
# ─── Pose coordinate range check ─────────────────────────────────────────────
pose3d_samples = [s for s in samples if s['pose_type'] == '3d']
pose2d_samples = [s for s in samples if s['pose_type'] == '2d']

if pose3d_samples:
    ex = pose3d_samples[0]['pose']
    print(f'pose_3d example — shape: {ex.shape}')
    print(f'  x range: [{ex[...,0].min():.3f}, {ex[...,0].max():.3f}]')
    print(f'  y range: [{ex[...,1].min():.3f}, {ex[...,1].max():.3f}]')
    print(f'  z range: [{ex[...,2].min():.3f}, {ex[...,2].max():.3f}]')

if pose2d_samples:
    ex = pose2d_samples[0]['pose']
    print(f'\npose_2d example — shape: {ex.shape}')
    print(f'  x range: [{ex[...,0].min():.3f}, {ex[...,0].max():.3f}]')
    print(f'  y range: [{ex[...,1].min():.3f}, {ex[...,1].max():.3f}]')

pose_3d example — shape: (451, 17, 3)
  x range: [-208.038, 693.327]
  y range: [-613.878, 281.936]
  z range: [-282.614, 1192.916]


## Section 4: Skeleton Feature Extraction

Identical logic to UTD-MHAD.  Only the joint-index constants differ.

### Joint mapping

| Semantic role      | UTD-MHAD (Kinect-20) | MM-Fit pose_3d (H3.6M-17) | MM-Fit pose_2d (COCO-18) |
|--------------------|----------------------|---------------------------|---------------------------|
| Head               | 0                    | 10                        | 0 (Nose)                  |
| Shoulder center    | 1                    | 8 (Thorax)                | 1 (Neck)                  |
| Spine              | 2                    | 7                         | 1 (Neck, approx)          |
| Hip center         | 3                    | 0 (Hip)                   | (avg L/R hip)             |
| Shoulder left      | 4                    | 14                        | 5                         |
| Elbow left         | 5                    | 15                        | 6                         |
| Wrist left         | 6                    | 16                        | 7                         |
| Hand left ≈ Wrist  | 7                    | 16 (approx)               | 7 (approx)                |
| Shoulder right     | 8                    | 11                        | 2                         |
| Elbow right        | 9                    | 12                        | 3                         |
| Wrist right        | 10                   | 13                        | 4                         |
| Hand right ≈ Wrist | 11                   | 13 (approx)               | 4 (approx)                |
| Hip left           | 12                   | 1                         | 11                        |
| Knee left          | 13                   | 2                         | 12                        |
| Ankle left         | 14                   | 3                         | 13                        |
| Foot left ≈ Ankle  | 15                   | 3 (approx)                | 13 (approx)               |
| Hip right          | 16                   | 4                         | 8                         |
| Knee right         | 17                   | 5                         | 9                         |
| Ankle right        | 18                   | 6                         | 10                        |
| Foot right ≈ Ankle | 19                   | 6 (approx)                | 10 (approx)               |

In [10]:
# =============================================================================
# Skeleton feature extractor — base class (identical to UTD-MHAD version)
# =============================================================================

class _SkeletonFeatureExtractorBase:
    """
    Base class with all feature-extraction logic.

    Subclasses set joint-index constants for their dataset; all feature
    computation references semantic names so vectors are identical in
    meaning and dimensionality across datasets.

    Feature categories (identical to UTD-MHAD pipeline):
      1. Normalized joint position statistics
      2. Pairwise joint distance statistics
      3. Joint angle statistics
      4. Bone (segment) length statistics
      5. Velocity statistics
      6. Acceleration statistics
      7. Covariance matrix features
      8. Centre-of-mass trajectory statistics
      9. Global motion energy

    Total output: 1879 features  (same as UTD-MHAD skeleton extractor)
    """

    # --- Subclasses MUST override these ---
    HEAD            = None
    SHOULDER_CENTER = None
    SPINE           = None
    HIP_CENTER      = None
    SHOULDER_LEFT   = None
    ELBOW_LEFT      = None
    WRIST_LEFT      = None
    HAND_LEFT       = None   # approximated by WRIST_LEFT where unavailable
    SHOULDER_RIGHT  = None
    ELBOW_RIGHT     = None
    WRIST_RIGHT     = None
    HAND_RIGHT      = None
    HIP_LEFT        = None
    KNEE_LEFT       = None
    ANKLE_LEFT      = None
    FOOT_LEFT       = None   # approximated by ANKLE_LEFT where unavailable
    HIP_RIGHT       = None
    KNEE_RIGHT      = None
    ANKLE_RIGHT     = None
    FOOT_RIGHT      = None

    NUM_JOINTS = None   # number of joints used in feature computation (=20)
    RAW_JOINTS = None   # joints in the raw data
    RAW_DIMS   = None   # coordinate dims (2 or 3)

    def __init__(self):
        self.JOINT_PAIRS = [
            (self.HAND_LEFT,     self.HAND_RIGHT),
            (self.FOOT_LEFT,     self.FOOT_RIGHT),
            (self.HAND_LEFT,     self.HIP_CENTER),
            (self.HAND_RIGHT,    self.HIP_CENTER),
            (self.HAND_LEFT,     self.HEAD),
            (self.HAND_RIGHT,    self.HEAD),
            (self.FOOT_LEFT,     self.HIP_CENTER),
            (self.FOOT_RIGHT,    self.HIP_CENTER),
            (self.WRIST_LEFT,    self.WRIST_RIGHT),
            (self.ELBOW_LEFT,    self.ELBOW_RIGHT),
            (self.SHOULDER_LEFT, self.SHOULDER_RIGHT),
            (self.KNEE_LEFT,     self.KNEE_RIGHT),
            (self.HIP_LEFT,      self.HIP_RIGHT),
            (self.HAND_LEFT,     self.FOOT_LEFT),
            (self.HAND_RIGHT,    self.FOOT_RIGHT),
            (self.HEAD,          self.HIP_CENTER),
        ]

        self.ANGLE_TRIPLETS = [
            (self.SHOULDER_LEFT,  self.ELBOW_LEFT,      self.WRIST_LEFT),
            (self.SHOULDER_RIGHT, self.ELBOW_RIGHT,     self.WRIST_RIGHT),
            (self.HIP_LEFT,       self.KNEE_LEFT,       self.ANKLE_LEFT),
            (self.HIP_RIGHT,      self.KNEE_RIGHT,      self.ANKLE_RIGHT),
            (self.SHOULDER_LEFT,  self.SHOULDER_CENTER, self.SHOULDER_RIGHT),
            (self.ELBOW_LEFT,     self.SHOULDER_LEFT,   self.SHOULDER_CENTER),
            (self.ELBOW_RIGHT,    self.SHOULDER_RIGHT,  self.SHOULDER_CENTER),
            (self.HIP_CENTER,     self.SPINE,           self.SHOULDER_CENTER),
            (self.HIP_LEFT,       self.HIP_CENTER,      self.HIP_RIGHT),
            (self.SHOULDER_LEFT,  self.SHOULDER_CENTER, self.HEAD),
            (self.SHOULDER_RIGHT, self.SHOULDER_CENTER, self.HEAD),
        ]

        self.BONE_PAIRS = [
            (self.HIP_CENTER,      self.SPINE),
            (self.SPINE,           self.SHOULDER_CENTER),
            (self.SHOULDER_CENTER, self.HEAD),
            (self.SHOULDER_CENTER, self.SHOULDER_LEFT),
            (self.SHOULDER_LEFT,   self.ELBOW_LEFT),
            (self.ELBOW_LEFT,      self.WRIST_LEFT),
            (self.WRIST_LEFT,      self.HAND_LEFT),
            (self.SHOULDER_CENTER, self.SHOULDER_RIGHT),
            (self.SHOULDER_RIGHT,  self.ELBOW_RIGHT),
            (self.ELBOW_RIGHT,     self.WRIST_RIGHT),
            (self.WRIST_RIGHT,     self.HAND_RIGHT),
            (self.HIP_CENTER,      self.HIP_LEFT),
            (self.HIP_LEFT,        self.KNEE_LEFT),
            (self.KNEE_LEFT,       self.ANKLE_LEFT),
            (self.ANKLE_LEFT,      self.FOOT_LEFT),
            (self.HIP_CENTER,      self.HIP_RIGHT),
            (self.HIP_RIGHT,       self.KNEE_RIGHT),
            (self.KNEE_RIGHT,      self.ANKLE_RIGHT),
            (self.ANKLE_RIGHT,     self.FOOT_RIGHT),
        ]

        self.KEY_JOINTS = [
            self.HIP_CENTER, self.SPINE, self.SHOULDER_CENTER, self.HEAD,
            self.SHOULDER_LEFT,  self.ELBOW_LEFT,  self.WRIST_LEFT,  self.HAND_LEFT,
            self.SHOULDER_RIGHT, self.ELBOW_RIGHT, self.WRIST_RIGHT, self.HAND_RIGHT,
            self.FOOT_LEFT, self.FOOT_RIGHT,
        ]

        self.COV_JOINTS = [
            self.HIP_CENTER, self.SPINE, self.SHOULDER_CENTER, self.HEAD,
            self.HAND_LEFT, self.HAND_RIGHT, self.FOOT_LEFT, self.FOOT_RIGHT,
        ]

    # ── Preprocessing ────────────────────────────────────────────────────────

    def _detect_missing(self, skel):
        """Returns boolean mask (frames, J); True = missing."""
        is_nan  = np.any(np.isnan(skel), axis=2)
        is_inf  = np.any(np.isinf(skel), axis=2)
        is_zero = np.all(np.abs(skel) < 1e-10, axis=2)
        return is_nan | is_inf | is_zero

    def _interpolate_missing(self, skel, missing_mask):
        """Linear temporal interpolation; forward/back fill at boundaries."""
        skel = skel.copy()
        n_frames, n_joints, n_dims = skel.shape

        for j in range(n_joints):
            if not np.any(missing_mask[:, j]):
                continue
            valid_idx = np.where(~missing_mask[:, j])[0]
            if len(valid_idx) == 0:
                continue
            for d in range(n_dims):
                skel[:, j, d] = np.interp(
                    np.arange(n_frames), valid_idx, skel[valid_idx, j, d])

        # Joints missing in ALL frames: copy from kinematic parent
        parent_map = {c: p for p, c in self.BONE_PAIRS}
        for j in np.where(np.all(missing_mask, axis=0))[0]:
            parent = parent_map.get(j)
            if parent is not None and not np.all(missing_mask[:, parent]):
                skel[:, j, :] = skel[:, parent, :]
            else:
                skel[:, j, :] = 0.0

        return skel

    def _preprocess(self, skel):
        """Detect missing → interpolate → Gaussian smooth."""
        skel         = np.nan_to_num(skel, nan=0.0, posinf=0.0, neginf=0.0)
        missing_mask = self._detect_missing(skel)
        n_missing    = np.sum(missing_mask)
        if n_missing > 0:
            logger.debug(f'Skeleton: {n_missing} missing joint-frames — interpolating')
            skel = self._interpolate_missing(skel, missing_mask)
        if skel.shape[0] >= 5:
            skel = gaussian_filter1d(skel, sigma=1.0, axis=0)
        return skel

    # ── Normalization ────────────────────────────────────────────────────────

    def _normalize_skeleton(self, skel):
        """Translate to hip center; scale by torso length (hip → shoulder center)."""
        hip      = skel[:, self.HIP_CENTER:self.HIP_CENTER + 1, :]
        skel_n   = skel - hip
        torso_v  = skel_n[:, self.SHOULDER_CENTER, :] - skel_n[:, self.HIP_CENTER, :]
        torso_l  = np.linalg.norm(torso_v, axis=1, keepdims=True)
        torso_l  = np.clip(torso_l, 1e-6, None)
        skel_n   = skel_n / torso_l[:, np.newaxis, :]
        return skel_n

    # ── Feature helpers ──────────────────────────────────────────────────────

    def _temporal_statistics(self, signal_2d):
        """9 statistics per column: mean, std, RMS, skew, kurtosis, range, median, Q1, Q3."""
        features = []
        for col in range(signal_2d.shape[1]):
            s = signal_2d[:, col]
            features.extend([
                np.mean(s),
                np.std(s),
                np.sqrt(np.mean(s ** 2)),
                stats.skew(s)     if len(s) > 2 else 0.0,
                stats.kurtosis(s) if len(s) > 3 else 0.0,
                np.max(s) - np.min(s),
                np.median(s),
                np.percentile(s, 25),
                np.percentile(s, 75),
            ])
        return np.array(features)

    def _compute_joint_distances(self, skel):
        """(frames, 16) pairwise distances."""
        return np.array([
            np.linalg.norm(skel[:, j1, :] - skel[:, j2, :], axis=1)
            for j1, j2 in self.JOINT_PAIRS
        ]).T

    def _compute_joint_angles(self, skel):
        """(frames, 11) joint angles."""
        angles = []
        for j1, j2, j3 in self.ANGLE_TRIPLETS:
            v1    = skel[:, j1, :] - skel[:, j2, :]
            v2    = skel[:, j3, :] - skel[:, j2, :]
            cos_a = np.sum(v1 * v2, axis=1) / (
                np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-8)
            angles.append(np.arccos(np.clip(cos_a, -1.0, 1.0)))
        return np.array(angles).T

    def _compute_bone_lengths(self, skel):
        """(frames, 19) bone lengths."""
        return np.array([
            np.linalg.norm(skel[:, j1, :] - skel[:, j2, :], axis=1)
            for j1, j2 in self.BONE_PAIRS
        ]).T

    def _compute_velocity(self, skel):
        if skel.shape[0] < 2:
            return np.zeros_like(skel)
        vel = np.diff(skel, axis=0)
        return np.vstack([vel, vel[-1:]])

    def _compute_acceleration(self, skel):
        vel = self._compute_velocity(skel)
        if vel.shape[0] < 2:
            return np.zeros_like(vel)
        acc = np.diff(vel, axis=0)
        return np.vstack([acc, acc[-1:]])

    def _covariance_features(self, skel):
        """Upper-triangle of covariance matrix over 8 key joints (n_dims × 8 coords)."""
        n_dims = skel.shape[2]
        flat   = skel.reshape(skel.shape[0], -1)
        idx    = []
        for j in self.COV_JOINTS:
            idx.extend([j * n_dims + d for d in range(n_dims)])
        flat_key = flat[:, idx]
        if flat_key.shape[0] < 2:
            cov = np.zeros((flat_key.shape[1], flat_key.shape[1]))
        else:
            cov = np.cov(flat_key.T)
        return cov[np.triu_indices(cov.shape[0])]

    # ── Shape helpers ────────────────────────────────────────────────────────

    def _ensure_3d_coords(self, skel):
        """
        Ensure skeleton has 3 coordinate dimensions.
        If 2D (frames, J, 2), pad z with zeros to get (frames, J, 3).
        """
        if skel.shape[2] == 3:
            return skel
        if skel.shape[2] == 2:
            z = np.zeros((*skel.shape[:2], 1), dtype=skel.dtype)
            return np.concatenate([skel, z], axis=2)
        return skel

    # ── Main extraction ──────────────────────────────────────────────────────

    def extract(self, skel):
        """
        Extract 1879-dim feature vector from a single skeleton sequence.

        Args:
            skel: (frames, J, dims)  — already sliced by loader
        Returns:
            np.ndarray of shape (1879,), or None on failure
        """
        if skel is None or skel.size == 0:
            return None
        if skel.ndim != 3 or skel.shape[0] == 0:
            return None

        # Ensure 3-D coordinates (pad 2-D with z=0)
        skel = self._ensure_3d_coords(skel)

        # Preprocess
        skel = self._preprocess(skel)

        # Normalize
        skel_norm = self._normalize_skeleton(skel)

        # 1. Position stats on key joints
        flat_pos = skel_norm.reshape(skel_norm.shape[0], -1)
        key_idx  = [j * 3 + d for j in self.KEY_JOINTS for d in range(3)]
        pos_feats  = self._temporal_statistics(flat_pos[:, key_idx])

        # 2. Joint distances
        dist_feats = self._temporal_statistics(self._compute_joint_distances(skel_norm))

        # 3. Joint angles
        angle_feats = self._temporal_statistics(self._compute_joint_angles(skel_norm))

        # 4. Bone lengths
        bone_feats = self._temporal_statistics(self._compute_bone_lengths(skel_norm))

        # 5. Velocity
        vel = self._compute_velocity(skel_norm).reshape(skel_norm.shape[0], -1)
        vel_feats = self._temporal_statistics(vel[:, key_idx])

        # 6. Acceleration
        acc = self._compute_acceleration(skel_norm).reshape(skel_norm.shape[0], -1)
        acc_feats = self._temporal_statistics(acc[:, key_idx])

        # 7. Covariance
        cov_feats = self._covariance_features(skel_norm)

        # 8. Centre of mass
        com = np.mean(skel_norm, axis=1)
        com_feats = self._temporal_statistics(com)

        # 9. Motion energy
        if skel_norm.shape[0] > 1:
            total_disp = np.sum(
                np.linalg.norm(np.diff(skel_norm, axis=0), axis=2), axis=1)
        else:
            total_disp = np.array([0.0])
        motion_energy = np.array([
            np.mean(total_disp), np.std(total_disp),
            np.max(total_disp),  np.sum(total_disp),
        ])

        all_feats = np.concatenate([
            pos_feats, dist_feats, angle_feats, bone_feats,
            vel_feats, acc_feats, cov_feats, com_feats, motion_energy,
        ])
        return np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)


logger.info('_SkeletonFeatureExtractorBase defined.')

2026-05-31 17:07:19,130 - INFO - _SkeletonFeatureExtractorBase defined.


In [11]:
# =============================================================================
# MM-Fit pose_3d extractor  (Human3.6M, 17 joints)
# =============================================================================

class SkeletonFeatureExtractor3D(_SkeletonFeatureExtractorBase):
    """
    Skeleton extractor for MM-Fit pose_3d  (Human3.6M keypoint model, 17 joints).

    Joint indices:
        0: Hip (center),  1: Left Hip,   2: Left Knee,   3: Left Foot,
        4: Right Hip,     5: Right Knee, 6: Right Foot,  7: Spine,
        8: Thorax,        9: Neck/Nose,  10: Head,       11: Right Shoulder,
       12: Right Elbow,  13: Right Wrist, 14: Left Shoulder,
       15: Left Elbow,   16: Left Wrist

    Notes:
      - No dedicated Hand or Foot joints → proxied by Wrist / Ankle.
      - Spine (joint 7) used for both SPINE and SHOULDER_CENTER proxy is avoided;
        Thorax (8) is the anatomical shoulder-center equivalent.
      - HIP_CENTER = 0 (the central hip joint in H3.6M).
    """
    HEAD            = 10
    SHOULDER_CENTER = 8    # Thorax
    SPINE           = 7
    HIP_CENTER      = 0    # Hip (center)
    SHOULDER_LEFT   = 14
    ELBOW_LEFT      = 15
    WRIST_LEFT      = 16
    HAND_LEFT       = 16   # proxy: same as wrist
    SHOULDER_RIGHT  = 11
    ELBOW_RIGHT     = 12
    WRIST_RIGHT     = 13
    HAND_RIGHT      = 13   # proxy
    HIP_LEFT        = 1
    KNEE_LEFT       = 2
    ANKLE_LEFT      = 3
    FOOT_LEFT       = 3    # proxy: same as ankle
    HIP_RIGHT       = 4
    KNEE_RIGHT      = 5
    ANKLE_RIGHT     = 6
    FOOT_RIGHT      = 6    # proxy

    NUM_JOINTS = 17
    RAW_JOINTS = 17
    RAW_DIMS   = 3


# =============================================================================
# MM-Fit pose_2d extractor  (COCO keypoint model, 18 joints)
# =============================================================================

class SkeletonFeatureExtractor2D(_SkeletonFeatureExtractorBase):
    """
    Skeleton extractor for MM-Fit pose_2d  (COCO keypoint model, 18 joints).

    Joint indices:
        0: Nose,           1: Neck,           2: Right Shoulder,
        3: Right Elbow,    4: Right Wrist,    5: Left Shoulder,
        6: Left Elbow,     7: Left Wrist,     8: Right Hip,
        9: Right Knee,    10: Right Ankle,   11: Left Hip,
       12: Left Knee,     13: Left Ankle,    14: Right Eye,
       15: Left Eye,      16: Right Ear,     17: Left Ear

    Notes:
      - No Hip Center joint → proxied by average of L/R hip (handled in
        _normalize_skeleton override).
      - No Spine joint → Neck (1) used as SPINE proxy.
      - No Hand/Foot → Wrist/Ankle proxied.
      - 2-D coords; z is padded with zeros in _ensure_3d_coords.
    """
    HEAD            = 0    # Nose
    SHOULDER_CENTER = 1    # Neck
    SPINE           = 1    # proxy: Neck
    HIP_CENTER      = 11   # proxy: Left Hip (see override below)
    SHOULDER_LEFT   = 5
    ELBOW_LEFT      = 6
    WRIST_LEFT      = 7
    HAND_LEFT       = 7
    SHOULDER_RIGHT  = 2
    ELBOW_RIGHT     = 3
    WRIST_RIGHT     = 4
    HAND_RIGHT      = 4
    HIP_LEFT        = 11
    KNEE_LEFT       = 12
    ANKLE_LEFT      = 13
    FOOT_LEFT       = 13
    HIP_RIGHT       = 8
    KNEE_RIGHT      = 9
    ANKLE_RIGHT     = 10
    FOOT_RIGHT      = 10

    NUM_JOINTS = 18
    RAW_JOINTS = 18
    RAW_DIMS   = 2

    def _normalize_skeleton(self, skel):
        """
        Override: compute virtual hip center as mean of L/R hip joints,
        then apply same torso-length normalization as base.
        """
        virtual_hip = (
            skel[:, self.HIP_LEFT:self.HIP_LEFT + 1, :]
            + skel[:, self.HIP_RIGHT:self.HIP_RIGHT + 1, :]
        ) / 2.0

        skel_n   = skel - virtual_hip
        # Torso: virtual hip → Neck
        torso_v  = skel_n[:, self.SHOULDER_CENTER, :] - virtual_hip[:, 0, :]
        torso_l  = np.linalg.norm(torso_v, axis=1, keepdims=True)
        torso_l  = np.clip(torso_l, 1e-6, None)
        skel_n   = skel_n / torso_l[:, np.newaxis, :]
        return skel_n


# Instantiate once
_skel_ext_3d = SkeletonFeatureExtractor3D()
_skel_ext_2d = SkeletonFeatureExtractor2D()


def extract_skeleton_features(pose, pose_type):
    """
    Route to the correct extractor based on pose_type ('3d' or '2d').
    Returns 1879-dim vector or None.
    """
    if pose is None:
        return None
    if pose_type == '3d':
        return _skel_ext_3d.extract(pose)
    if pose_type == '2d':
        return _skel_ext_2d.extract(pose)
    return None


logger.info('Skeleton extractors defined (3D: H3.6M-17, 2D: COCO-18).')

2026-05-31 17:07:19,145 - INFO - Skeleton extractors defined (3D: H3.6M-17, 2D: COCO-18).


## Section 5: Sensor Feature Extraction

Identical to UTD-MHAD inertial feature extractor.
Applied independently to each available sensor stream, then concatenated.

In [12]:
# =============================================================================
# Sensor Feature Extractor
# (identical logic to InertialFeatureExtractor in UTD-MHAD notebook)
# =============================================================================

class SensorFeatureExtractor:
    """
    Extracts features from a single sensor stream.

    Supports:
      - Accelerometer / Gyroscope / Magnetometer: (N, data_cols) where
        data_cols = [Timestamp, X, Y, Z]  → uses cols 1:4 (3 channels)
      - Heart-rate: (N, data_cols) where data_cols = [Timestamp, HR]
        → uses col 1 (1 channel)

    Features per stream (time-domain + frequency-domain, window-aggregated):
      - Per channel: mean, std, RMS, skewness, kurtosis, power, range,
        median, MAV, Q1, Q3, IQR, zero-crossing rate, peak count,
        dominant freq, spectral entropy, 3 band energies  → 17 features
      - For 3-channel streams: magnitude channel features + SMA +
        cross-axis correlations (3 within-sensor + acc-gyro)
    """

    def __init__(self, config):
        self.window_size = config.SENSOR_WINDOW_SIZE
        self.overlap     = config.SENSOR_WINDOW_OVERLAP

    # ── signal primitives ────────────────────────────────────────────────────

    def _sma(self, data):
        return np.mean(np.sum(np.abs(data), axis=1))

    def _signal_power(self, s):
        return np.mean(s ** 2)

    def _zero_crossing_rate(self, s):
        sc = s - np.mean(s)
        return np.sum(np.abs(np.diff(np.sign(sc)))) / (2.0 * len(s))

    def _peak_count(self, s):
        peaks, _ = signal.find_peaks(s)
        return len(peaks) / max(len(s), 1)

    def _spectral_entropy(self, s, fs=50):
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        psd_n = psd / (np.sum(psd) + 1e-12)
        psd_n = psd_n[psd_n > 0]
        return -np.sum(psd_n * np.log2(psd_n + 1e-12))

    def _dominant_frequency(self, s, fs=50):
        if len(s) < 4:
            return 0.0
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        return freqs[np.argmax(psd)]

    def _band_energy(self, s, fs=50, bands=[(0, 5), (5, 15), (15, 25)]):
        if len(s) < 4:
            return [0.0] * len(bands)
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        return [np.sum(psd[(freqs >= lo) & (freqs < hi)]) for lo, hi in bands]

    def _channel_features(self, s):
        """17 features for a single channel."""
        return [
            np.mean(s),
            np.std(s),
            np.sqrt(np.mean(s ** 2)),
            stats.skew(s)     if len(s) > 2 else 0.0,
            stats.kurtosis(s) if len(s) > 3 else 0.0,
            self._signal_power(s),
            np.max(s) - np.min(s),
            np.median(s),
            np.mean(np.abs(s)),
            np.percentile(s, 25),
            np.percentile(s, 75),
            np.percentile(s, 75) - np.percentile(s, 25),
            self._zero_crossing_rate(s),
            self._peak_count(s),
            self._dominant_frequency(s),
            self._spectral_entropy(s),
            *self._band_energy(s),
        ]

    # ── window extraction ────────────────────────────────────────────────────

    def _extract_window_features_multichannel(self, window):
        """
        For a 3-channel (acc/gyr/mag) window: (W, 3).
        Mirrors UTD-MHAD InertialFeatureExtractor._extract_window_features.
        """
        features = []
        for ch in range(window.shape[1]):
            features.extend(self._channel_features(window[:, ch]))
        # Magnitude channel
        mag = np.linalg.norm(window, axis=1)
        features.extend(self._channel_features(mag))
        # SMA
        features.append(self._sma(window))
        # Cross-axis correlations (3 pairs)
        for i in range(window.shape[1]):
            for j in range(i + 1, window.shape[1]):
                if len(window[:, i]) > 1:
                    c = np.corrcoef(window[:, i], window[:, j])[0, 1]
                    features.append(c if not np.isnan(c) else 0.0)
                else:
                    features.append(0.0)
        return features

    def _extract_window_features_single(self, window):
        """For a 1-channel (HR) window: (W,) or (W, 1)."""
        s = window.ravel()
        return self._channel_features(s)

    # ── main extract ─────────────────────────────────────────────────────────

    def extract(self, data, is_single_channel=False):
        """
        Extract feature vector from a full sensor sequence.

        Args:
            data: (N, data_cols) where data_cols include Timestamp + channels.
                  If 3-channel sensor: data_cols = [Timestamp, X, Y, Z]
                    → data[:, 1:4] used.
                  If HR: data_cols = [Timestamp, HR]
                    → data[:, 1] used.
            is_single_channel: True for HR streams.

        Returns:
            1D numpy array (aggregated window features) or None.
        """
        if data is None or data.shape[0] == 0:
            return None

        data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

        # Extract channels (drop Timestamp column)
        if is_single_channel or data.shape[1] == 1:
            channels = data[:, -1:]   # (N, 1)
        else:
            channels = data[:, 1:]    # (N, 3)

        # Low-pass Butterworth filter
        try:
            b, a = signal.butter(4, 0.2, btype='low')
            for ch in range(channels.shape[1]):
                if len(channels[:, ch]) > 12:
                    channels[:, ch] = signal.filtfilt(b, a, channels[:, ch])
        except Exception:
            pass

        n = channels.shape[0]
        step = max(int(self.window_size * (1 - self.overlap)), 1)

        windows = [
            channels[start: start + self.window_size]
            for start in range(0, n - self.window_size + 1, step)
        ]
        if not windows:
            windows = [channels]

        single_ch = (channels.shape[1] == 1 or is_single_channel)
        win_feats = []
        for w in windows:
            if single_ch:
                win_feats.append(self._extract_window_features_single(w))
            else:
                win_feats.append(self._extract_window_features_multichannel(w))

        win_feats = np.array(win_feats)

        if win_feats.shape[0] == 1:
            return np.nan_to_num(win_feats[0], nan=0.0, posinf=0.0, neginf=0.0)

        aggregated = []
        for col in range(win_feats.shape[1]):
            d = win_feats[:, col]
            aggregated.extend([np.mean(d), np.std(d), np.min(d), np.max(d)])

        return np.nan_to_num(np.array(aggregated), nan=0.0, posinf=0.0, neginf=0.0)


def extract_all_sensor_features(sensor_streams, config):
    """
    Extract features from all available sensor streams and concatenate.

    Returns:
        1D numpy array (concatenation of all stream features), or None if
        no stream produced a feature vector.
    """
    extractor   = SensorFeatureExtractor(config)
    hr_streams  = set(config.HR_STREAMS)
    all_streams = config.SENSOR_STREAMS + config.HR_STREAMS + config.MAG_STREAMS

    parts = []
    expected_dim = None

    for stream in all_streams:
        data = sensor_streams.get(stream)
        is_hr = stream in hr_streams

        # If missing → create zero placeholder
        if data is None or data.shape[0] == 0:
            if expected_dim is None:
                # create dummy to determine size
                dummy = np.zeros((config.SENSOR_WINDOW_SIZE, 2 if is_hr else 4))
                feat = extractor.extract(dummy, is_single_channel=is_hr)
                expected_dim = len(feat)
            parts.append(np.zeros(expected_dim))
            continue

        feat  = extractor.extract(data, is_single_channel=is_hr)

        if expected_dim is None:
            expected_dim = len(feat)

        # pad if still different
        if len(feat) != expected_dim:
            feat = np.pad(feat, (0, expected_dim - len(feat)))

        parts.append(feat)

    return np.concatenate(parts)


logger.info('SensorFeatureExtractor defined.')

2026-05-31 17:07:19,163 - INFO - SensorFeatureExtractor defined.


## Section 6: Multimodal Feature Extraction Pipeline

In [13]:
# =============================================================================
# MM-Fit Multimodal HAR Pipeline
# =============================================================================

class MMFitHARPipeline:
    """
    Loads data, extracts features, fuses modalities, optionally classifies.
    Mirrors the structure of MultimodalHARPipeline from UTD-MHAD.
    """

    def __init__(self, config):
        self.config = config
        self.scaler = StandardScaler()

    # ── feature extraction ───────────────────────────────────────────────────

    def extract_features(self, samples):
        """
        Extract and fuse features for all samples.

        Returns:
            list of dicts, each containing:
                features        → concatenated feature vector
                label           → activity_class
                participant     → participant ID
                workout_id      → workout folder string
                modality_features → {'skeleton': arr, 'sensor': arr}
                modality_sizes    → {'skeleton': int, 'sensor': int}
        """
        results = []
        n = len(samples)

        for i, s in enumerate(samples):
            if (i + 1) % 100 == 0 or i == 0:
                logger.info(f'Extracting features: {i+1}/{n}')

            parts = {}

            # Skeleton features
            skel_feat = extract_skeleton_features(s['pose'], s['pose_type'])
            if skel_feat is not None:
                parts['skeleton'] = skel_feat

            # Sensor features
            sensor_feat = extract_all_sensor_features(s['sensor_streams'], self.config)
            if sensor_feat is not None:
                parts['sensor'] = sensor_feat

            if not parts:
                logger.debug(f'Sample {i} ({s["workout_id"]}) has no usable features — skipped.')
                continue

            concatenated = np.concatenate(list(parts.values()))
            results.append({
                'features':         concatenated,
                'label':            s['activity_class'],
                'participant':      s['participant'],
                'workout_id':       s['workout_id'],
                'modality_features': parts,
                'modality_sizes':    {k: v.shape[0] for k, v in parts.items()},
            })

        logger.info(f'Extracted features for {len(results)} / {n} samples.')

        if results:
            ex = results[0]
            logger.info(f'Total feature vector dimension: {ex["features"].shape[0]}')
            for mod, sz in ex['modality_sizes'].items():
                logger.info(f'  {mod}: {sz} features')

        return results

    # ── save ──────────────────────────────────────────────────────────────────

    def save_features(self, results):
        """
        Save extracted features to disk (mirrors UTD-MHAD save_features).

        Saves:
          X_feat.pkl       — list of dicts: {skeleton_feat, sensor_feat}
          y.npy            — encoded activity labels
          participants.npy — participant IDs
          label_encoder.pkl
        """
        out_dir = self.config.FEATURES_DIR
        os.makedirs(out_dir, exist_ok=True)

        X_feat       = []
        y_raw        = []
        participants = []

        for r in results:
            mf = r['modality_features']
            X_feat.append({
                'skeleton_feat': mf.get('skeleton', np.array([])),
                'sensor_feat':   mf.get('sensor',   np.array([])),
            })
            y_raw.append(r['label'])
            participants.append(r['participant'])

        y_raw        = np.array(y_raw)
        participants = np.array(participants)

        le = LabelEncoder()
        y  = le.fit_transform(y_raw)

        joblib.dump(X_feat, os.path.join(out_dir, 'X_feat.pkl'))
        np.save(os.path.join(out_dir, 'y.npy'), y)
        np.save(os.path.join(out_dir, 'participants.npy'), participants)
        joblib.dump(le, os.path.join(out_dir, 'label_encoder.pkl'))

        logger.info(f"Features saved to '{out_dir}':")
        logger.info(f'  X_feat.pkl:         {len(X_feat)} samples')
        if X_feat:
            first = X_feat[0]
            for k in ['skeleton_feat', 'sensor_feat']:
                dim = first[k].shape[0] if first[k].size > 0 else 0
                logger.info(f'    {k}: {dim}')
        logger.info(f'  y.npy:              {y.shape}  ({len(le.classes_)} classes)')
        logger.info(f'  participants.npy:   {participants.shape}  '
                     f'IDs={sorted(np.unique(participants).tolist())}')
        logger.info(f'  label classes:      {le.classes_.tolist()}')
        return le

    # ── split ────────────────────────────────────────────────────────────────

    def split_train_test(self, results):
        """Split by participant using the config's TRAIN/TEST lists."""
        X_tr, y_tr = [], []
        X_te, y_te = [], []

        for r in results:
            pid = r['participant']
            if pid in self.config.TRAIN_PARTICIPANTS:
                X_tr.append(r['features'])
                y_tr.append(r['label'])
            elif pid in self.config.TEST_PARTICIPANTS:
                X_te.append(r['features'])
                y_te.append(r['label'])

        X_tr = np.array(X_tr);  y_tr = np.array(y_tr)
        X_te = np.array(X_te);  y_te = np.array(y_te)

        logger.info(f'Train: {X_tr.shape[0]} samples | Test: {X_te.shape[0]} samples')
        if X_tr.shape[0]:
            logger.info(f'Feature dimension: {X_tr.shape[1]}')
            logger.info(f'Classes: {sorted(np.unique(y_tr).tolist())}')

        return X_tr, y_tr, X_te, y_te

    # ── classify ────────────────────────────────────────────────────────────

    def train_and_evaluate(self, X_tr, y_tr, X_te, y_te):
        """Train Random Forest; print evaluation metrics."""
        X_tr_s = self.scaler.fit_transform(X_tr)
        X_te_s = self.scaler.transform(X_te)
        X_tr_s = np.nan_to_num(X_tr_s, nan=0.0, posinf=0.0, neginf=0.0)
        X_te_s = np.nan_to_num(X_te_s, nan=0.0, posinf=0.0, neginf=0.0)

        cfg = self.config
        rf  = RandomForestClassifier(
            n_estimators=cfg.RF_N_ESTIMATORS,
            max_depth=cfg.RF_MAX_DEPTH,
            min_samples_split=cfg.RF_MIN_SAMPLES_SPLIT,
            min_samples_leaf=cfg.RF_MIN_SAMPLES_LEAF,
            random_state=cfg.RF_RANDOM_STATE,
            n_jobs=cfg.RF_N_JOBS,
            class_weight='balanced',
        )

        logger.info('Training Random Forest …')
        t0 = time.time()
        rf.fit(X_tr_s, y_tr)
        logger.info(f'Training done in {time.time()-t0:.2f}s')

        y_pred = rf.predict(X_te_s)
        acc    = accuracy_score(y_te, y_pred)
        f1_mac = f1_score(y_te, y_pred, average='macro',    zero_division=0)
        f1_wt  = f1_score(y_te, y_pred, average='weighted', zero_division=0)

        logger.info(f'\n{"="*60}')
        logger.info('RESULTS')
        logger.info(f'{"="*60}')
        logger.info(f'Accuracy:         {acc*100:.2f}%')
        logger.info(f'Macro F1:         {f1_mac*100:.2f}%')
        logger.info(f'Weighted F1:      {f1_wt*100:.2f}%')

        unique_labels = sorted(np.unique(np.concatenate([y_te, y_pred])))
        target_names  = [f'Class {l}' for l in unique_labels]
        report = classification_report(
            y_te, y_pred, labels=unique_labels,
            target_names=target_names, digits=3, zero_division=0)
        logger.info(f'\nClassification Report:\n{report}')

        logger.info('Running 5-fold CV on training set …')
        cv = cross_val_score(rf, X_tr_s, y_tr, cv=5, scoring='accuracy', n_jobs=-1)
        logger.info(f'CV Accuracy: {cv.mean()*100:.2f}% (±{cv.std()*100:.2f}%)')

        importances = rf.feature_importances_
        top_k = min(20, len(importances))
        top_idx = np.argsort(importances)[-top_k:][::-1]
        logger.info(f'\nTop {top_k} feature importances:')
        for idx in top_idx:
            logger.info(f'  Feature {idx}: {importances[idx]:.4f}')

        return {
            'accuracy': acc, 'f1_macro': f1_mac, 'f1_weighted': f1_wt,
            'cv_mean':  cv.mean(), 'cv_std': cv.std(),
            'model': rf,
        }

    # ── run ──────────────────────────────────────────────────────────────────

    def run(self, samples):
        """Full pipeline: extract → save → split → train → evaluate."""
        logger.info(f'\n{"="*60}')
        logger.info('MM-FIT MULTIMODAL HAR PIPELINE')
        logger.info(f'{"="*60}')

        logger.info('\n--- Step 1: Extracting features ---')
        results = self.extract_features(samples)

        if not results:
            logger.error('No features extracted!')
            return None

        logger.info('\n--- Step 2: Saving features ---')
        self.save_features(results)

        logger.info('\n--- Step 3: Train / test split ---')
        X_tr, y_tr, X_te, y_te = self.split_train_test(results)

        if X_tr.shape[0] == 0 or X_te.shape[0] == 0:
            logger.error('Empty train or test set — check participant split.')
            return None

        logger.info('\n--- Step 4: Train & evaluate ---')
        return self.train_and_evaluate(X_tr, y_tr, X_te, y_te)


logger.info('MMFitHARPipeline defined.')

2026-05-31 17:07:19,183 - INFO - MMFitHARPipeline defined.


## Section 7: Run the Pipeline

### Step 0: Structured Gaussian Noise (before feature extraction)

Add Gaussian noise to **%** of raw data per modality **before** feature extraction:
- **Skeleton**: noise on % of joints (all 3 coords × all frames for selected joints)
- **Inertial**: noise on % of sensor channels (all timesteps for selected channels)

Noise std per unit = `NOISE_STRENGTH × std` of that unit's own data (`NOISE_STRENGTH=0.20`).
Applied in-place on `samples` immediately before `pipeline.run(samples)`.

In [14]:
import gc
import numpy as np
from tqdm import tqdm

NOISE_FRACTION = 0.50
NOISE_STRENGTH = 1.0
NOISE_SEED     = 42

ALL_INERTIAL_STREAMS = config.SENSOR_STREAMS + config.HR_STREAMS + config.MAG_STREAMS


def apply_structured_noise(samples, fraction=NOISE_FRACTION,
                            strength=NOISE_STRENGTH, seed=NOISE_SEED):
    """
    Apply structured Gaussian noise IN-PLACE to skeleton and inertial data.

    Skeleton  (frames, joints, dims):
        - Randomly select `fraction` of joint indices.
        - For each selected joint, add N(0, strength × joint_std) noise
          to ALL frames and ALL coordinate dims for that joint.

    Inertial  (timesteps, channels):  applies to every sensor stream
        - Randomly select `fraction` of channel indices (excl. timestamp col).
        - For each selected channel, add N(0, strength × channel_std) noise
          to ALL timesteps for that channel.

    Returns
    -------
    samples : same list, modified in-place
    stats   : dict with cumulative noise statistics
    """
    rng = np.random.RandomState(seed)

    total_skel_joints_noised     = 0
    total_skel_joints_possible   = 0
    total_iner_channels_noised   = 0
    total_iner_channels_possible = 0
    samples_with_skel  = 0
    samples_with_iner  = 0

    for idx, s in enumerate(tqdm(samples, desc="Structured Gaussian noise")):

        # ── Skeleton ──────────────────────────────────────────────────────
        pose = s.get('pose')
        if pose is not None and pose.ndim == 3 and pose.shape[0] > 0:
            n_joints = pose.shape[1]           # (frames, joints, dims)
            n_noisy  = max(1, int(round(n_joints * fraction)))
            joints_to_noise = rng.choice(n_joints, size=n_noisy, replace=False)
            for j in joints_to_noise:
                jdata = pose[:, j, :]          # (frames, dims)
                sigma = jdata.std() * strength
                if sigma > 0:
                    pose[:, j, :] += rng.normal(0, sigma, size=jdata.shape)

            total_skel_joints_noised   += n_noisy
            total_skel_joints_possible += n_joints
            samples_with_skel          += 1

        # ── Inertial (all sensor streams) ─────────────────────────────────
        sensor_streams = s.get('sensor_streams', {})
        for stream_name in ALL_INERTIAL_STREAMS:
            data = sensor_streams.get(stream_name)
            if data is None or data.ndim < 2 or data.shape[0] == 0:
                continue

            # data shape: (timesteps, cols); col 0 = Timestamp, cols 1+ = channels
            n_data_cols = data.shape[1] - 1
            if n_data_cols <= 0:
                continue

            n_noisy   = max(1, int(round(n_data_cols * fraction)))
            ch_offset = 1                      # channels start at index 1
            channels_to_noise = rng.choice(n_data_cols, size=n_noisy, replace=False) + ch_offset
            for ch in channels_to_noise:
                ch_data = data[:, ch]
                sigma   = ch_data.std() * strength
                if sigma > 0:
                    data[:, ch] += rng.normal(0, sigma, size=ch_data.shape)

            total_iner_channels_noised   += n_noisy
            total_iner_channels_possible += n_data_cols
            samples_with_iner            += 1

        if (idx + 1) % 200 == 0:
            gc.collect()

    stats = {
        'samples_with_skeleton':         samples_with_skel,
        'skeleton_joints_noised':         total_skel_joints_noised,
        'skeleton_joints_possible':       total_skel_joints_possible,
        'samples_with_inertial_streams':  samples_with_iner,
        'inertial_channels_noised':       total_iner_channels_noised,
        'inertial_channels_possible':     total_iner_channels_possible,
    }
    return samples, stats


# ── Run noise injection ───────────────────────────────────────────────────────
print(f"Applying structured Gaussian noise "
      f"(fraction={NOISE_FRACTION:.0%}, strength={NOISE_STRENGTH:.0%} × std, "
      f"seed={NOISE_SEED}) ...")
samples, noise_stats = apply_structured_noise(samples)

# ── Report ────────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("STRUCTURED GAUSSIAN NOISE SUMMARY")
print("=" * 60)

s_possible = noise_stats['skeleton_joints_possible']
s_noised   = noise_stats['skeleton_joints_noised']
i_possible = noise_stats['inertial_channels_possible']
i_noised   = noise_stats['inertial_channels_noised']

print(f"  Skeleton samples processed : {noise_stats['samples_with_skeleton']}")
if s_possible > 0:
    avg_joints_total  = s_possible / max(noise_stats['samples_with_skeleton'], 1)
    avg_joints_noised = s_noised   / max(noise_stats['samples_with_skeleton'], 1)
    print(f"  Avg joints per sample      : {avg_joints_total:.1f}")
    print(f"  Avg joints noised          : {avg_joints_noised:.1f}  "
          f"({100*s_noised/s_possible:.1f}% of all joint slots)")

print()
print(f"  Inertial stream calls      : {noise_stats['samples_with_inertial_streams']}")
if i_possible > 0:
    avg_ch_total  = i_possible / max(noise_stats['samples_with_inertial_streams'], 1)
    avg_ch_noised = i_noised   / max(noise_stats['samples_with_inertial_streams'], 1)
    print(f"  Avg channels per stream    : {avg_ch_total:.1f}")
    print(f"  Avg channels noised        : {avg_ch_noised:.1f}  "
          f"({100*i_noised/i_possible:.1f}% of all channel slots)")

print()
print(f"  TOTAL skeleton joints noised  : {s_noised:,}  /  {s_possible:,}")
print(f"  TOTAL inertial channels noised: {i_noised:,}  /  {i_possible:,}")
print(f"  Noise strength                : {NOISE_STRENGTH:.0%} × per-unit std")
print("=" * 60)
print("Noise injection complete. Proceeding to feature extraction.")


Applying structured Gaussian noise (fraction=50%, strength=100% × std, seed=42) ...


Structured Gaussian noise: 100%|██████████| 616/616 [00:01<00:00, 485.34it/s]


STRUCTURED GAUSSIAN NOISE SUMMARY
  Skeleton samples processed : 616
  Avg joints per sample      : 17.0
  Avg joints noised          : 8.0  (47.1% of all joint slots)

  Inertial stream calls      : 6469
  Avg channels per stream    : 2.7
  Avg channels noised        : 1.9  (68.4% of all channel slots)

  TOTAL skeleton joints noised  : 4,928  /  10,472
  TOTAL inertial channels noised: 12,013  /  17,557
  Noise strength                : 100% × per-unit std
Noise injection complete. Proceeding to feature extraction.


In [15]:
pipeline = MMFitHARPipeline(config)
results  = pipeline.run(samples)

2026-05-31 17:07:20,488 - INFO - 
2026-05-31 17:07:20,489 - INFO - MM-FIT MULTIMODAL HAR PIPELINE
2026-05-31 17:07:20,489 - INFO - ============================================================
2026-05-31 17:07:20,490 - INFO - 
--- Step 1: Extracting features ---
2026-05-31 17:07:20,490 - INFO - Extracting features: 1/616
2026-05-31 17:15:17,032 - INFO - Extracting features: 100/616
2026-05-31 17:23:27,899 - INFO - Extracting features: 200/616
2026-05-31 17:31:52,989 - INFO - Extracting features: 300/616
2026-05-31 17:40:34,497 - INFO - Extracting features: 400/616
2026-05-31 17:49:25,605 - INFO - Extracting features: 500/616
2026-05-31 17:58:06,721 - INFO - Extracting features: 600/616
2026-05-31 17:59:41,278 - INFO - Extracted features for 616 / 616 samples.
2026-05-31 17:59:41,279 - INFO - Total feature vector dimension: 6359
2026-05-31 17:59:41,280 - INFO -   skeleton: 1879 features
2026-05-31 17:59:41,280 - INFO -   sensor: 4480 features
2026-05-31 17:59:41,281 - INFO - 
--- Step 2:

## Section 8: Load Saved Features (Offline Use)

Run this section independently after Section 7 has been executed at least once.

In [16]:
import joblib, numpy as np, os

out_dir = config.FEATURES_DIR

X_feat       = joblib.load(os.path.join(out_dir, 'X_feat.pkl'))
y            = np.load(os.path.join(out_dir, 'y.npy'))
participants = np.load(os.path.join(out_dir, 'participants.npy'))
le           = joblib.load(os.path.join(out_dir, 'label_encoder.pkl'))

print(f'Loaded {len(X_feat)} samples')
print(f'y shape:            {y.shape}   classes: {le.classes_.tolist()}')
print(f'participants shape:  {participants.shape}')
if X_feat:
    first = X_feat[0]
    print(f'skeleton_feat dim:  {first["skeleton_feat"].shape}')
    print(f'sensor_feat dim:    {first["sensor_feat"].shape}')

Loaded 616 samples
y shape:            (616,)   classes: ['bicep_curls', 'dumbbell_rows', 'dumbbell_shoulder_press', 'jumping_jacks', 'lateral_shoulder_raises', 'lunges', 'pushups', 'situps', 'squats', 'tricep_extensions']
participants shape:  (616,)
skeleton_feat dim:  (1879,)
sensor_feat dim:    (4480,)
